# Open Quantum System Market Microstructure Replication
### Research Paper: *A Quantum Canonical Framework for Order Book Dynamics: Proper-Time Relaxation in Non-Equilibrium Markets*
**Author:** Sanghyeok Lee (Hilbert Labs)  
**Official DOI:** [10.5281/zenodo.23042820](https://doi.org/10.5281/zenodo.23042820) | **Zenodo Record:** [https://zenodo.org/records/23042820](https://zenodo.org/records/23042820)  
**Course Reference:** Udemy Section 8 Masterclass Replication Notebook

---

This notebook reproduces the fundamental analytical and numerical results presented in the paper:
1. **The Financial Planck Constant ($)** and phase-space lattice quantization.
2. **SU(2) Unitary Evolution** and trace-preservation proofs.
3. **Lindblad Dissipative Dynamics** and population pumping.
4. **Volume-Clocked Proper-Time Evolution (\tau$)** and intrinsic timescale contraction.
5. **Entropy Drop Phase Transitions & Golden Windows**.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sys, os
sys.path.insert(0, os.path.abspath('..'))

from core.engine import QuantumCanonicalEngine
from core.planck_lattice import PlanckLattice
from core.canonical_hamiltonian import CanonicalHamiltonianBuilder
from core.proper_time import ProperTimeClock, GoldenWindowPhaseGater

print('Core Quantum Canonical Microstructure Package Loaded Successfully!')


## 1. Planck Lattice Discretization & Microstructure Hamiltonian
Quantize Level-2 order book depth and construct the 2-level Hamiltonian:
\hat{H} = \begin{pmatrix} V_{\text{norm}}^+ & i \Delta \\ i \Delta & V_{\text{norm}}^- \end{pmatrix}


In [ ]:
bids = [(20500.00, 15.0), (20499.75, 25.0), (20499.50, 40.0), (20499.25, 60.0), (20499.00, 100.0)]
asks = [(20500.25, 10.0), (20500.50, 18.0), (20500.75, 30.0), (20501.00, 45.0), (20501.25, 70.0)]

lattice = PlanckLattice(tick_size=0.25, lot_size=1.0)
bids_q, asks_q, mid = lattice.quantize_order_book(bids, asks)

print(f'Financial Planck Constant (h_f): {lattice.h_f} pt*lot')
print(f'Mid-Price Reference: {mid}')
print('Quantized Bids (Tick Offset, Lot Depth):\n', bids_q)
print('Quantized Asks (Tick Offset, Lot Depth):\n', asks_q)


## 2. Dynamic Proper-Time Simulation Across Microstructure Regimes
- **Calm Phase (0~100)**: Symmetric liquidity, low flux (Ground state).
- **Quench Shock (100~150)**: Large aggressive buy volume consuming asks.
- **Relaxation (150~300)**: Dephasing and return towards mixed state.


In [ ]:
np.random.seed(42)
engine = QuantumCanonicalEngine(tick_size=0.25, mass=2.0)
history = []

for step in range(300):
    if step < 100:
        buy_vol, sell_vol, dt = np.random.exponential(1.5), np.random.exponential(1.5), 0.5
        bids, asks = [(20500.00, 20.0)], [(20500.25, 20.0)]
    elif 100 <= step < 150:
        buy_vol, sell_vol, dt = np.random.exponential(45.0), np.random.exponential(5.0), 0.05
        bids, asks = [(20500.50, 50.0)], [(20500.75, 5.0)]
    else:
        buy_vol, sell_vol, dt = np.random.exponential(3.0), np.random.exponential(3.0), 0.3
        bids, asks = [(20500.75, 25.0)], [(20501.00, 25.0)]
        
    res = engine.process_orderbook_snapshot(
        bids=bids, asks=asks,
        trade_buy_volume=buy_vol, trade_sell_volume=sell_vol,
        dt_physical_sec=dt, utc_hour_float=13.35 if (100 <= step < 150) else 13.80
    )
    history.append(res)

df = pd.DataFrame(history)
print(f'Simulation finished: {len(df)} steps.')


## 3. Visualization: Information Purity, Entropy Drop, & Dwell Dynamics


In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(10, 8), sharex=True)
axes[0].plot(df['purity'], color='navy', lw=2, label='Quantum Purity ||r||')
axes[0].plot(df['r_z'], color='crimson', lw=1.5, linestyle='--', label='Polarization r_z')
axes[0].axvspan(100, 150, color='orange', alpha=0.25, label='Quench Shock')
axes[0].set_ylabel('Bloch Vector')
axes[0].legend(loc='upper left')
axes[0].grid(True, linestyle=':')

axes[1].plot(df['entropy'], color='purple', lw=2, label='Von Neumann Entropy S(rho)')
axes[1].plot(df['info_purity'], color='teal', lw=1.5, label='Info Purity P_info')
axes[1].axvspan(100, 150, color='orange', alpha=0.25)
axes[1].set_ylabel('Entropy (nats)')
axes[1].legend(loc='upper left')
axes[1].grid(True, linestyle=':')

axes[2].plot(df['dtau'], color='forestgreen', lw=1.5, label='Proper Time (dtau)')
axes[2].plot(df['tau_relax'], color='black', lw=2, label='Dwell Scale (tau)')
axes[2].axvspan(100, 150, color='orange', alpha=0.25)
axes[2].set_ylabel('Proper Time Scale')
axes[2].set_xlabel('Event Index (k)')
axes[2].legend(loc='upper left')
axes[2].grid(True, linestyle=':')

plt.tight_layout()
plt.show()
